# Orient'IA — EDA et entraînement ML

Classification multi-classes : profil → **16 parcours ISPM** (catalogue issu de `corpusISPMNote`).

**Modèles comparés :** baseline (régression logistique), Random Forest, HistGradientBoosting.

Ce notebook importe les modules du package `orientia` (même logique que les CLI).

In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd
import matplotlib.pyplot as plt

# Racine projet (au cas où le kernel n'a pas le package éditable)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from orientia.config import SYNTHETIC_CSV, TARGET_COL, FEATURE_COLS, COMPARISON_REPORT_PATH
from orientia.data.generate_synthetic import generate_synthetic, save_synthetic
from orientia.ml.evaluate import eda_summary, class_imbalance_note
from orientia.ml.train import train_and_compare

pd.set_option("display.max_columns", 40)

## 1. Génération / chargement des données synthétiques

In [ ]:
from orientia.data.process_corpus import process_corpus

process_corpus()  # catalogue + chunks à jour

if not SYNTHETIC_CSV.exists():
    df, meta = generate_synthetic(n=3000, seed=42)
    save_synthetic(df)
    print(meta)
else:
    df = pd.read_csv(SYNTHETIC_CSV)
    print(f"Chargé {len(df)} profils / {df[TARGET_COL].nunique()} parcours depuis {SYNTHETIC_CSV}")

df.head()

## 2. Analyse exploratoire

In [ ]:
summary = eda_summary(df, TARGET_COL)
print("Distribution des labels:", summary["label_distribution"])
print("Déséquilibre:", class_imbalance_note(df[TARGET_COL]))

ax = df[TARGET_COL].value_counts().plot(kind="bar", title="Répartition des parcours (synthétique)")
ax.set_xlabel("Parcours")
ax.set_ylabel("Effectif")
plt.tight_layout()
plt.show()

df[FEATURE_COLS[:7]].describe()

## 3. Entraînement et comparaison des modèles

Si l'enquête (`responses_anonymized.csv`) contient assez de lignes, elle sert de jeu de test.
Sinon : hold-out 20 % synthétique.

In [ ]:
report = train_and_compare()
print("Meilleur modèle:", report["best_model"], "| macro_f1 =", round(report["best_macro_f1"], 4))
print("Source test:", report["test_source"])

rows = []
for name, entry in report["models"].items():
    t = entry["test"]
    rows.append({
        "model": name,
        "accuracy": t["accuracy"],
        "macro_f1": t["macro_f1"],
        "weighted_f1": t["weighted_f1"],
        "top_2_accuracy": t.get("top_2_accuracy"),
    })
pd.DataFrame(rows).sort_values("macro_f1", ascending=False)

## 4. Limites

- Données d'entraînement synthétiques dérivées du corpus → risque de retrouver les centroïdes.
- Sans enquête réelle, la généralisation vers profils humains n'est pas mesurée.
- 16 parcours ISPM ; détail corpus inégal (IGGLIA riche, ICMP peu documenté).
- Chunks RAG dans `data/corpus/corpus_chunks.jsonl` (prêts pour l'assistant).